In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only ".
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Imports

In [2]:
import os
import librosa
from torch.utils.data import Dataset,DataLoader,random_split
from kaggle_secrets import UserSecretsClient
import wandb
from sklearn.metrics import f1_score
import torch 
import torch.nn as nn
import torch.optim as optim

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("login")
os.environ["WANDB_API_KEY"]=wandb_api
wandb.login()
print("Wandb login successful")

wandb: Currently logged in as: 23f1000559 (23f1000559-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb login successful


## Dataset & loader

In [4]:
!ls /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00000

bass.wav  drums.wav  other.wav	vocals.wav


In [5]:
#dataset
class MyDataset(Dataset):
    def __init__(self,root_dir,sr=16000):
        self.files=[]
        self.labels = []
        self.sr = sr
        self.stems=['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
        for label, folder in enumerate(os.listdir(root_dir)):
            folder_path = os.path.join(root_dir,folder)
            for file in os.listdir(folder_path):
                self.files.append(os.path.join(folder_path,file))
                self.labels.append(label)
    def __len__(self):
        return len(self.files)
    def __getitem__(self,idx):
        mixed=None
        for stem in self.stems:
            stem_path = os.path.join(self.files[idx],stem)
            audio,_ = librosa.load(stem_path,sr=self.sr)
            if mixed is None:
                mixed = audio
            else:
                min_len = min(len(mixed),len(audio))
                mixed = mixed[:min_len]+ audio[:min_len]
        # 10s audio length
        max_len = self.sr *10
        if len(mixed)<max_len:
            mixed = np.pad(mixed,(0,max_len-len(mixed)))
        else:
            mixed = mixed[:max_len]
        mel = librosa.feature.melspectrogram(
            y=mixed,n_mels=128,sr=self.sr
        )
        mel_db = librosa.power_to_db(mel)
        return torch.tensor(mel_db).unsqueeze(0), self.labels[idx]
        
        

In [6]:
root_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

In [7]:
#dataset instance
dataset = MyDataset(root_dir)
# train val split
train_size = int(0.8* len(dataset))
val_size = len(dataset)- train_size
train_dataset,val_dataset = random_split(dataset,[train_size,val_size])

In [8]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

## Model

In [9]:
# Start a new wandb run 
run = wandb.init(
    # Set the wandb project where this run will be logged.
    project="23f1000559-t12026",
    entity ="23f1000559-indian-institute-of-technology-madras",
    # Track hyperparameters and run metadata.
    config={
        "epochs": 1,
        "batch_size": 32,
        "learning_rate": 0.0001,
        "model": "custom-cnn-v1"
    },
)

wandb: setting up run xera7a5y
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260321_133719-xera7a5y
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fine-snow-29
wandb: ⭐️ View project at https://wandb.ai/23f1000559-indian-institute-of-technology-madras/23f1000559-t12026
wandb: 🚀 View run at https://wandb.ai/23f1000559-indian-institute-of-technology-madras/23f1000559-t12026/runs/xera7a5y


In [10]:
## cnn architecture
class cnn(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1,32,kernel_size=3)
        self.c2 = nn.Conv2d(32,64,kernel_size=2)
        self.bn = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout(0.5)
        self.ap = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(64,100)
        self.fc2 = nn.Linear(100,10)
        self.relu = nn.ReLU()
    def forward(self,x):
        x = self.c1(x)
        x = self.c2(x)
        x = self.relu(x)
        x = self.bn(x)
        x = self.pool(x)
        x = self.drop(x)
        x = self.ap(x)
        x = torch.flatten(x,1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


In [11]:
# initalising model , loss , optimiser
device = "cuda" if torch.cuda.is_available() else "cpu"
model = cnn().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.0001)

## Training Loop

In [12]:
num_epochs = 5
for epoch in range(num_epochs):
    total_loss = 0
    model.train()
    for x,y in train_loader:
        x =x.to(device)
        y = y.to(device)
        output=model(x)
        loss = criterion(output,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss +=loss.item()
    avg_loss = loss/len(train_loader)
    #Log loss to wandb
    wandb.log({"epoch": epoch+1, "train_loss": avg_loss})
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 0.0909
Epoch 2/5, Loss: 0.0844
Epoch 3/5, Loss: 0.0880
Epoch 4/5, Loss: 0.0864
Epoch 5/5, Loss: 0.0834


## Validation Loop

In [13]:
model.eval()
total_val_loss =0
all_preds=[]
all_labels = []
with torch.no_grad():
    for x_val,y_val in val_loader:
        x_val = x_val.to(device)
        y_val = y_val.to(device)
        output= model(x_val)
        loss = criterion(output, y_val)
        total_val_loss += loss.item()
        # store for f1
        all_preds.extend(output.argmax(dim=1).cpu().numpy())
        all_labels.extend(y_val.cpu().numpy())
avg_val_loss = total_val_loss / len(val_loader)
val_macro_f1 = f1_score(all_labels,all_preds,average="macro")
print(f"Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
#log to wandb
wandb.log({"val_loss": avg_val_loss, "val_macro_f1": val_macro_f1})
        

Val Loss: 2.0599, Val Macro F1: 0.2179


## Prediction

In [14]:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

In [15]:
model.eval()
predicted_genres=[]
root = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
for file_path in test_df["filename"]:
    final_path = os.path.join(root,file_path)
    audio,_ = librosa.load(final_path,sr=16000)
    mel = librosa.feature.melspectrogram(
            y=audio,n_mels=128,sr=16000
        )
    mel_db = librosa.power_to_db(mel)
    tensor = torch.tensor(mel_db).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.no_grad():
        output = model(tensor)
        pred = output.argmax(dim=1).item()
        predicted_genres.append(pred)
        

In [16]:
print(list(enumerate(os.listdir("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"))))

[(0, 'disco'), (1, 'metal'), (2, 'reggae'), (3, 'blues'), (4, 'rock'), (5, 'classical'), (6, 'jazz'), (7, 'hiphop'), (8, 'country'), (9, 'pop')]


## Submission

In [17]:
genre_labels=['disco','metal','reggae','blues','rock','classical','jazz','hiphop','country','pop']

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": [genre_labels[i] for i in predicted_genres]
})
submission.to_csv("submission.csv", index=False)
print(submission.head())

   id      genre
0   1     hiphop
1   2  classical
2   3    country
3   4      metal
4   5    country
